<a id="intro"></a>
# ЛР № 5. Расчёт метрик городского велопроката Сеула

Первая часть общего аналитического ноутбука ЛР № 5 и № 7.
Здесь читаем готовый `mart`, рассчитываем метрики и проверяем результаты.
В седьмой работе ниже добавляются
визуализации и исследование. Промежуточные файлы данных не создаются.

**Основной вопрос:** Как различается спрос на аренду велосипедов по сезонам и часам суток?

Все обязательные описания, определения и вывод собраны в
[docs/metrics.md](../docs/metrics.md).


## 1. Вход и параметры

Используем проектное ядро `.venv` и существующий S3-стенд.
Повторный запуск всей первой части восстанавливает результаты после перезапуска ядра.

`source_ingested_on=2026-11-12` - логическая дата загрузки исходного снимка.
Записи аренды относятся к периоду `2017-12-01` - `2018-11-30`.

In [1]:
DATASET_SLUG = "seoul_bike"
SOURCE_INGESTED_ON = "2026-11-12"
MART_PATH = f"s3://mart/{DATASET_SLUG}/source_ingested_on={SOURCE_INGESTED_ON}/rent.parquet"
PERIOD_START = "2017-12-01"
PERIOD_END = "2018-12-01" # Период не включительно
print(f"Вход: {MART_PATH}")
print(f"Период по Date: [{PERIOD_START}, {PERIOD_END})")


Вход: s3://mart/seoul_bike/source_ingested_on=2026-11-12/rent.parquet
Период по Date: [2017-12-01, 2018-12-01)


## 2. Подключение

Переиспользуем подключение из предыдущих работ. Настройки читаются из `.env`.


In [5]:
import os
import sys
from html import escape
from pathlib import Path

import duckdb
from dotenv import load_dotenv
from IPython.display import HTML, display


PROJECT_ROOT = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "pyproject.toml").is_file()
    and (parent / "src/analysis/read_raw.py").is_file()
)
load_dotenv(PROJECT_ROOT / ".env", override=True)
settings = ("S3_ENDPOINT", "S3_ACCESS_KEY", "S3_SECRET_KEY")
missing = [name for name in settings if not os.getenv(name)]
if missing:
    raise RuntimeError(f"Не заданы настройки: {', '.join(missing)}")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from analysis.read_raw import configure_s3
connection = duckdb.connect()
configure_s3(connection)


def literal(value):
    return "'" + value.replace("'", "''") + "'"


def select_table(sql):
    cursor = connection.execute(sql)
    description = cursor.description
    if description is None:
        raise ValueError("Ожидается таблица результата")
    rows = cursor.fetchall()
    header = "".join(f"<th>{escape(column[0])}</th>" for column in description)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(str(item))}</td>" for item in row) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))
    print(f"Строк результата: {len(rows)}")
    return rows

## 3. Чтение витрины

Одна строка `rent` — один час работы системы велопроката: дата, час, число арендованных велосипедов, сезон и признак рабочего дня. Бизнес-ключ `(date, hour)` уникален.

Общий фильтр задаёт только отчётный период. Нерабочие часы пока не исключаются:
их нули отражают недоступность сервиса, а не отсутствие спроса, и это будет учтено
в определениях метрик в следующих разделах.

`analysis_base` один раз загружает выбранные данные из S3 в память DuckDB.
Следующие расчёты и будущие графики ЛР № 7 используют это же подключение.


In [5]:
connection.execute(f"""
    CREATE OR REPLACE TEMP VIEW rent AS
    SELECT * FROM read_parquet({literal(MART_PATH)}, hive_partitioning = false)
""")

required_fields = {
    "date", "hour", "seasons", "rented_bike_count",
    "functioning_day", "functioning_day_violation",
}
schema = {row[0]: row[1] for row in connection.execute("DESCRIBE rent").fetchall()}
assert not (required_fields - schema.keys()), "В витрине отсутствуют нужные поля"

connection.execute(f"""
    CREATE OR REPLACE TEMP TABLE analysis_base AS
    SELECT * FROM rent
    WHERE date >= DATE {literal(PERIOD_START)}
      AND date < DATE {literal(PERIOD_END)}
""")

input_rows = select_table("""
    SELECT count(*) AS records_count,
           count(*) FILTER (WHERE functioning_day) AS working_count,
           count(*) FILTER (WHERE NOT functioning_day) AS non_working_count,
           sum(rented_bike_count) AS total_rentals,
           min(date) AS first_date,
           max(date) AS last_date
    FROM analysis_base
""")

print(f"Физических полей витрины: {len(schema)}")

records_count,working_count,non_working_count,total_rentals,first_date,last_date
8760,8465,295,6172314,2017-12-01,2018-11-30


Строк результата: 1
Физических полей витрины: 6


## 4. Что считаем

Три метрики:
- число записей,
- суммарное число аренд,
- среднее число аренд в час.

Среднее число аренд рассчитывается только по рабочим часам (`functioning_day = true`). Нерабочие часы остаются для подсчёта общего количества записей и входят в общую сумму (где они дают ноль), но исключаются из расчёта среднего, чтобы недоступность сервиса не искажала интенсивность спроса.

Полные определения и пояснение окна находятся в `docs/metrics.md`.

## 5. Общий итог

Фильтр рабочих часов находится у конкретных агрегатов, поскольку общий `WHERE` на признак дня
изменил бы одновременно и количество, и сумму.

In [10]:
connection.execute("""
    CREATE OR REPLACE TEMP TABLE overall_metrics AS
    SELECT count(*) AS records_count,
        count(*) FILTER (WHERE functioning_day) AS working_count,
        sum(rented_bike_count) AS total_rentals,
        avg(rented_bike_count) FILTER (WHERE functioning_day) AS avg_rentals
    FROM analysis_base
""")
overall_rows = select_table("""
                            SELECT records_count,
                                   working_count,
                                   total_rentals,
                                   round(avg_rentals, 2) AS avg_rentals
                            FROM overall_metrics
                            """)

overall_raw = connection.execute("SELECT * FROM overall_metrics").fetchone()
assert overall_raw is not None

records_count,working_count,total_rentals,avg_rentals
8760,8465,6172314,729.16


Строк результата: 1


## 6. Сравнение по часам и окно

Одна строка результата - час суток за все дни выбранного года.
Окно добавляет долю группы в общем количестве аренд, сохраняя 24 часовые строки.

In [11]:
connection.execute("""
    CREATE OR REPLACE TEMP TABLE hourly_metrics AS
    WITH grouped AS (
        SELECT hour,
            count(*) AS records_count,
            count(*) FILTER (WHERE functioning_day) AS working_count,
            sum(rented_bike_count) AS total_rentals,
            avg(rented_bike_count) FILTER (WHERE functioning_day) AS avg_rentals
        FROM analysis_base
        GROUP BY hour
    )
    SELECT *,
           100.0 * total_rentals
              / nullif(sum(total_rentals) OVER (), 0) AS percent_of_rentals
    FROM grouped
""")
hourly_rows = select_table("""
                           SELECT hour,
                                  records_count,
                                  working_count,
                                  total_rentals,
                                  round(avg_rentals, 2) AS avg_rentals,
                                  round(percent_of_rentals, 3) AS percent_of_rentals
                           FROM hourly_metrics
                           ORDER BY hour
                           """)
hourly_raw = connection.execute("""
                                SELECT *
                                FROM hourly_metrics
                                ORDER BY hour
                                """).fetchall()

hour,records_count,working_count,total_rentals,avg_rentals,percent_of_rentals
0,365,352,197633,561.46,3.202
1,365,352,155557,441.92,2.52
2,365,352,110095,312.77,1.784
3,365,352,74216,210.84,1.202
4,365,352,48396,137.49,0.784
5,365,352,50765,144.22,0.822
6,365,352,104961,298.18,1.701
7,365,353,221192,626.61,3.584
8,365,353,370731,1050.23,6.006
9,365,353,235784,667.94,3.82


Строк результата: 24


## 7. Уточнение и отдельные значения

Добавляем сезон к группировке по часам, чтобы проверить, сохраняется ли суточный профиль спроса внутри разных сезонов. 
Временная таблица содержит все имеющиеся сочетания сезона и часа, а также размеры групп и знаменатели для средних.

Отдельная выборка содержит исходные значения числа аренд только для рабочих часов. 
Она нужна для построения распределения в ЛР №7 (нерабочие часы исключены, чтобы нули недоступности сервиса не искажали картину). 
Восстановить это распределение из средних значений по часам невозможно.

In [1]:
connection.execute("""
    CREATE OR REPLACE TEMP TABLE segment_metrics AS
    SELECT seasons, hour,
        count(*) AS records_count,
        count(*) FILTER (WHERE functioning_day) AS working_count,
        sum(rented_bike_count) AS total_rentals,
        avg(rented_bike_count) FILTER (WHERE functioning_day) AS avg_rentals
    FROM analysis_base 
    GROUP BY seasons, hour
""")

print("Первые 10 строк группировки по сезону и часу:")
segment_preview = select_table("""
    SELECT seasons, hour, records_count, working_count,
           round(avg_rentals, 2) AS avg_rentals
    FROM segment_metrics
    ORDER BY seasons, hour
    LIMIT 10
""")

# Уточняющая группировка, сохраняется в память
segment_rows = connection.execute("""
    SELECT * FROM segment_metrics ORDER BY seasons, hour
""").fetchall()
print(f"Строк сезонной группировки: {len(segment_rows)}")

# Выборка отдельных значений
connection.execute("""
    CREATE OR REPLACE TEMP TABLE distribution_values AS
    SELECT rented_bike_count FROM analysis_base WHERE functioning_day
""")

rental_summary = select_table("""
    SELECT count(*) AS records_count,
           min(rented_bike_count) AS min_rentals,
           median(rented_bike_count) AS median_rentals,
           max(rented_bike_count) AS max_rentals
    FROM distribution_values
""")[0]

# Список значений для будущего построения графика
rentals = [row[0] for row in connection.execute("""
    SELECT rented_bike_count FROM distribution_values ORDER BY rented_bike_count
""").fetchall()]

NameError: name 'connection' is not defined

<a id="validation"></a>
## 8. Проверка расчётов

Проверяем количество, число подходящих сумм и правильное объединение средних.
Для денежного среднего используем веса по `cost_count`, а не по числу всех строк.
Числа с плавающей точкой сравниваются с небольшим допуском. Краткие объяснения
ожидаемых и фактических результатов записаны в `docs/metrics.md`.


In [21]:
# M07. Проверки
check_rows = select_table("""
                          WITH checks AS (SELECT 1                           AS check_no,
                                                 'Сумма количества по часам' AS check_name,
                                                 (SELECT sum(records_count) FROM hourly_metrics)::DOUBLE AS actual, (SELECT records_count FROM overall_metrics) ::DOUBLE AS expected
                                          UNION ALL
                                          SELECT 2,
                                                 'Сумма количества по районам и часам',
                                                 (SELECT sum(records_count) FROM segment_metrics),
                                                 (SELECT records_count FROM overall_metrics)
                                          UNION ALL
                                          SELECT 3,
                                                 'Количество сумм для средней стоимости',
                                                 (SELECT sum(cost_count) FROM segment_metrics),
                                                 (SELECT count(total_amount) FROM analysis_base WHERE total_amount >= 0)
                                          UNION ALL
                                          SELECT 4,
                                                 'Средняя стоимость совпадает при расчёте по записям и по часовым итогам',
                                                 (SELECT sum(cost_sum) / nullif(sum(cost_count), 0)
                                                  FROM hourly_metrics),
                                                 (SELECT avg_amount FROM overall_metrics))
                          SELECT check_name,
                                 round(actual, 2)              AS actual,
                                 round(expected, 2)            AS expected,
                                 abs(actual - expected) < 1e-4 AS passed
                          FROM checks
                          ORDER BY check_no
                          """)
assert all(row[3] for row in check_rows), "Проверки не сошлись: вернитесь к расчётам"
print(f"Проверок пройдено: {len(check_rows)} из {len(check_rows)}")


check_name,actual,expected,passed
Сумма количества по часам,48235.0,48235.0,True
Сумма количества по районам и часам,48235.0,48235.0,True
Количество сумм для средней стоимости,48083.0,48083.0,True
Средняя стоимость совпадает при расчёте по записям и по часовым итогам,22.75,22.75,True


Строк результата: 4
Проверок пройдено: 4 из 4


<a id="continuation"></a>
## Продолжение в ЛР № 7

Первая часть завершена. Пояснения и численный вывод находятся в
[docs/metrics.md](../docs/metrics.md). Подготовлены временные таблицы `hourly_metrics`,
`segment_metrics`, `distribution_values`; общий итог - в `overall_metrics`.
Для работы из Python доступны `overall_raw`, `hourly_raw`, `segment_rows`,
`segment_totals`, `durations` и `duration_summary`. Эти данные не округлены.
`hourly_rows` и `overall_rows` служат для чтения округлённого вывода.

В седьмой работе ниже будут добавлены ячейки визуализации и исследования.
После перезапуска ядра сначала выполните M01-M07: все результаты должны
восстанавливаться из кода, а не зависеть от случайно сохранившейся памяти.
Изменив условия в расчётном блоке, повторите зависимые ячейки и проверки,
затем обновите описания в `metrics.md` и последующие графики.
